# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m09_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 교관 평가 점수와 AI 숙련도 점수 간 편차 분석 및 가중치 재튜닝
**소요시간**: 약 3.0시간 | **필수 산출물**: docs/trn01_aar_report.md (필수), configs/trn01_tuned_params.yaml (필수)

### 📋 미션 개요 (Overview)
초기 가중치를 실제 기수 훈련 데이터에 적용하면 야간·돌풍 조건이나 특정 위험 사건에서 교관 평가와 AI 점수 간 편차가 발생할 수 있다. 본 미션에서는 사후강평(AAR) 관점으로 편차를 조건별·사건별·습관별로 분해해 원인을 규명하고, 기수 내 교차검증 격자 탐색으로 채점 파라미터를 재튜닝(`trn01_tuned_params.yaml`)한다. 학습에 쓰지 않은 신규 기수 2개로 회귀 테스트를 수행해 개선 효과와 반복 불일치 특성을 보고서(`trn01_aar_report.md`)에 기록한다.

### 🎓 학습 목표 (Objectives)
초기 모델이 실제 기수 환경에 투입된 뒤 발생하는 성능 저하를 **사후강평(AAR) 관점에서 원인 분해하고 교차검증으로 재보정하는 MLOps 개선 사이클 역량**을 익힌다. 조건·위험 사건·습관별 점수 편차를 분해해 과적합·환경 편향 원인을 진단하고, 격자 탐색으로 재튜닝 파라미터를 도출한 뒤 미학습 신규 기수 2개로 회귀 테스트해 일반화 성능을 검증하는 절차를 배운다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션9 코드를 실행해 초기 AI의 일치율과 편차 원인 분해를 확인한다.
#### 이 단계의 역할
미션9 코드를 실행해 초기 AI의 일치율과 편차 원인 분해를 확인한다.

#### 해야 할 일

- 미션9 코드를 실행해 초기 AI의 일치율과 편차 원인 분해를 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 초기 일치율 47.5%, 돌풍 비행 평균 -40점 과소평가 원인 규명 재튜닝 후 평균 +42.9%p 향상, 회귀 테스트 2/2 통과

#### 학습자가 제출해야 하는 내용

- 미션9 코드를 실행해 초기 AI의 일치율과 편차 원인 분해를 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 주간 비행 기준으로 맞춘 초기 AI를 전체 기수 데이터에 적용했을 때 발생하는 교관 점수와의 편차를 조건별·위험 사건 수별·습관별로 분해하는 AAR 분석 스크립트를 작성해줘.
**AI 응답 검증 포인트**

- 편차 분해 합계와 전체 오차의 방향성이 논리적으로 일치하는지 확인
- 단순 평균 오차(MAE)뿐만 아니라 특정 조건(예: 야간+돌풍)의 극단적 편향이 드러나는지 점검

#### Step 2. 격자 탐색 상위 조합과 선택된 재튜닝값을 확인한다.
#### 이 단계의 역할
격자 탐색 상위 조합과 선택된 재튜닝값을 확인한다.

#### 해야 할 일

- 격자 탐색 상위 조합과 선택된 재튜닝값을 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 초기 일치율 47.5%, 돌풍 비행 평균 -40점 과소평가 원인 규명 재튜닝 후 평균 +42.9%p 향상, 회귀 테스트 2/2 통과

#### 학습자가 제출해야 하는 내용

- 격자 탐색 상위 조합과 선택된 재튜닝값을 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 특징 구성, 조건 표준화 여부, 규제 강도를 기수 내 교차검증(Grid Search)으로 탐색해 상위 5개 파라미터 조합을 비교하고 최종 재튜닝값을 YAML로 저장해줘.
**AI 응답 검증 포인트**

- 교차검증 분할 시 동일 조종자의 비행이 학습과 검증 폴드에 동시에 섞이지 않았는지(GroupKFold 등) 확인
- 저장된 trn01_tuned_params.yaml 파일의 키와 값이 모델 로더 규격과 일치하는지 검증

#### Step 3. 새 기수 2개 회귀 테스트 결과와 반복 불일치의 특성을 정리한다.
#### 이 단계의 역할
새 기수 2개 회귀 테스트 결과와 반복 불일치의 특성을 정리한다.

#### 해야 할 일

- 새 기수 2개 회귀 테스트 결과와 반복 불일치의 특성을 정리한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 초기 일치율 47.5%, 돌풍 비행 평균 -40점 과소평가 원인 규명 재튜닝 후 평균 +42.9%p 향상, 회귀 테스트 2/2 통과

#### 학습자가 제출해야 하는 내용

- 새 기수 2개 회귀 테스트 결과와 반복 불일치의 특성을 정리한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 학습에 전혀 쓰지 않은 신규 기수 2개 데이터로 재튜닝 모델을 회귀 테스트하고, 여전히 교관과 불일치하는 반복 사례의 공통 특성을 정리해줘.
**AI 응답 검증 포인트**

- 신규 기수 2개에서 초기 모델 대비 등급 일치율과 점수 오차가 일관되게 개선됐는지 확인
- 반복 불일치 사례가 모델 한계인지 교관 평가 기준 모호성인지 AAR 관점으로 분류

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
격자에서 조건 표준화를 False로 고정하면 어떤 조합이 선택되고 돌풍 비행 편차가 얼마나 남는지 확인한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 재튜닝 과정에서 과적합을 방지하기 위해 규제 강도(alpha)를 변화시켰을 때 기수 내 검증 점수와 신규 기수 회귀 테스트 점수 간의 차이를 비교해줘.
**AI 응답 검증 포인트**

- 검증 점수와 신규 기수 테스트 점수 격차가 줄어드는 적정 규제 구간을 수치로 확인
- 규제 강화로 핵심 위험 사건 감점 특징까지 무력화되지 않았는지 점검


In [ ]:
"""AAR(사후검토)로 교관 평가 점수와 AI 숙련도 점수의 편차를 비행 조건·위험 사건·조작 습관별로 분석해 원인을 규명하고, 특징 구성·조건 표준화·가중치(규제 강도)를 격자 탐색으로 재튜닝한 뒤 다른 조종자 집단으로 회귀 테스트해 개선 전후를 비교한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

# ---------------------------------------------------------------------------
# [교육용가상자료] 드론 조종 시뮬레이터 비행로그 생성기 (실제 훈련생·장비 정보 없음, 조종자는 가명 P001~)
# 비행 1회 = 120초(10Hz): 경로 비행(0~60초) -> 호버링(60~85초) -> 돌발 장애물 회피(85초) -> 비상 착륙(100초 지시)
# 조종자마다 숨은 숙련도(skill)와 조작 습관(급조작 jerky, 과보정 overcorr)이 있고, 교관 점수는 이를 반영해 만든다.
# ---------------------------------------------------------------------------
DT, STEPS = 0.1, 1200
T = np.arange(STEPS) * DT
COND = {"day": dict(wind=0.4, gust=0.0, gps=0.25, delay=0.0), "night": dict(wind=0.4, gust=0.0, gps=0.5, delay=0.15),
        "gust": dict(wind=1.2, gust=2.5, gps=0.25, delay=0.0)}
OBST_T, LAND_T = 85.0, 100.0


def make_pilots(n=40, seed=7):
    rng = np.random.default_rng(seed)
    skill = rng.beta(2.2, 2.0, n)
    return pd.DataFrame({"pilot": [f"P{i + 1:03d}" for i in range(n)], "skill": skill,
                         "jerky": rng.random(n) < 0.15 + 0.35 * (1 - skill), "overcorr": rng.random(n) < 0.15 + 0.35 * (1 - skill)})


def target_path(t):
    """사각형 경로(40x30m, 고도 20m)를 3m/s로 비행 -> 60초 지점에서 호버링 -> 착륙 지시 후 고도 0."""
    # [TODO : 학생 실습 9] AAR 기반 교관 평가 점수와 AI 숙련도 점수 간 편차 분석 및 가중치 재튜닝 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def fly(pilots, cond="day", seed=0, delay_extra=0.0, obstacle_side=None):
    """여러 조종자의 비행 1회를 동시에 모의한다(벡터화). 반환: 로그 배열 dict, 비행별 정답 정보 DataFrame."""
    # [TODO : 학생 실습 9] AAR 기반 교관 평가 점수와 AI 숙련도 점수 간 편차 분석 및 가중치 재튜닝 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def one(log, j):
    """여러 조종자 로그에서 j번째 조종자 1명의 로그(DataFrame)를 꺼낸다."""
    return pd.DataFrame({k: (v if v.ndim == 1 else v[:, j]) for k, v in log.items()})
# ---------------------------------------------------------------------------

# ---- 비행로그 특징 추출 공통 함수(여러 조종자 로그를 한 번에 계산) ----
from scipy.ndimage import uniform_filter1d

BASE_F = ["path_rmse", "hover_drift", "hover_std", "alt_err"]
ADV_F = BASE_F + ["obs_rt", "obs_lateral", "touch_vz", "att_exceed", "jerk", "overshoot", "n_risk"]
RULES = {"near_collision": ("obs_lateral", "<", 2.5), "hard_landing": ("touch_vz", ">", 2.0), "attitude_exceed": ("att_exceed", ">=", 10)}


def detect(df):
    """위험 조작 사건 탐지 규칙: 장애물 2.5초 후 회피 거리 2.5m 미만, 접지 속도 2.0m/s 초과, 자세 35도 초과 1초 이상."""
    out = {}
    for ev, (f, op, th) in RULES.items():
        out[ev] = df[f] < th if op == "<" else (df[f] > th if op == ">" else df[f] >= th)
    return pd.DataFrame(out)


def features(log):
    """경로 이탈도·호버링 안정성(기본) + 장애물 반응·착륙 속도·자세 한계·급조작·과보정(고도화) 특징."""
    X, Y, A = (uniform_filter1d(log[k], 9, axis=0) for k in ("x", "y", "alt"))       # GPS 잡음 완화(이동평균 0.9초)
    h = np.hypot(X - log["tx"], Y - log["ty"])
    cr, hv = slice(0, 600), slice(620, 850)
    k0, k1 = int(OBST_T / DT), int((OBST_T + 2.5) / DT)
    side = np.sign(log["ty"][k0 + 1] - log["ty"][k0 - 1])
    moved = (log["stick_roll"][k0:k0 + 40] * side) > 0.3
    rt = np.where(moved.any(0), moved.argmax(0) * DT, 4.0)
    vz = -np.gradient(A, DT, axis=0)
    land = np.argmax((A < 0.4) & (T[:, None] >= LAND_T), axis=0)
    touch = np.array([vz[max(i - 6, 0):i - 2, j].mean() if i > 6 else 3.0 for j, i in enumerate(land)])
    ds = np.abs(np.diff(np.stack([log["stick_roll"], log["stick_pitch"]]), axis=1)) / DT
    return pd.DataFrame({
        "path_rmse": np.sqrt((h[cr] ** 2).mean(0)), "hover_drift": h[hv].mean(0), "hover_std": np.hypot(X[hv].std(0), Y[hv].std(0)),
        "alt_err": np.abs(A[:850] - log["talt"][:850]).mean(0),
        "obs_rt": rt, "obs_lateral": np.abs(Y[k1] - Y[k0]), "touch_vz": touch,
        "att_exceed": (np.maximum(np.abs(log["roll_deg"]), np.abs(log["pitch_deg"])) > 35).sum(0),
        "jerk": np.percentile(ds, 95, axis=(0, 1)),
        "overshoot": np.clip(np.abs(Y[k0:k0 + 120] - Y[k0]).max(0) - 6.0, 0, None)}).pipe(
        lambda d: d.assign(n_risk=detect(d).sum(axis=1)))                                     # overshoot: 회피 후 목표 6m를 넘어선 거리(과보정)


def make_dataset(n_pilots=40, seed=7, conds=("day", "night", "gust"), flights=1):
    """조종자 n명 x 조건 x 반복 비행 -> 비행별 특징 + 정답(교관 점수·등급·위험 사건) 표."""
    pilots = make_pilots(n_pilots, seed)
    out = []
    for c in conds:
        for f in range(flights):
            log, tr = fly(pilots, c, seed=seed * 100 + list(COND).index(c) * 10 + f)
            out.append(pd.concat([tr.reset_index(drop=True), features(log)], axis=1).assign(flight=f))
    return pd.concat(out, ignore_index=True)


def cond_z(df, ref, cols):
    """조건별 기준 집단(ref)의 평균·표준편차로 표준화: 돌풍처럼 어려운 조건에서 받는 불이익을 줄인다."""
    z = df.copy()
    z[cols] = z[cols].astype(float)
    for c in df["cond"].unique():
        r = ref[ref["cond"] == c][cols]
        z.loc[df["cond"] == c, cols] = (df.loc[df["cond"] == c, cols] - r.mean()) / (r.std() + 1e-9)
    return z


def to_grade(score):
    return np.where(score >= 75, "A", np.where(score >= 55, "B", "C"))


def fit(df, feats, norm, alpha):
    X = cond_z(df, df, feats)[feats] if norm else df[feats]
    return Ridge(alpha=alpha).fit(X, df["instructor"])


def predict(m, df, ref, feats, norm):
    X = cond_z(df, ref, feats)[feats] if norm else df[feats]
    return np.clip(m.predict(X), 0, 100)


# 초기 운용 AI: 주간 비행만으로 맞춘 Baseline(경로·호버링·고도, 조건 표준화 없음)
day_train = make_dataset(40, seed=7, conds=("day",))
INIT = dict(feats=BASE_F, norm=False, alpha=1.0)
init_m = fit(day_train, **INIT)

# 1) 무엇이 일어났나: 훈련 1개 기수(조종자 40명 x 3조건)의 교관 점수와 AI 점수 비교
aar = make_dataset(40, seed=8)
aar["ai"] = predict(init_m, aar, day_train, INIT["feats"], INIT["norm"])
aar["dev"] = aar["ai"] - aar["instructor"]
acc0 = (to_grade(aar["ai"]) == aar["grade"]).mean()
print(f"[1. 무엇이 일어났나] 초기 AI 등급 일치율 {acc0:.1%}, 점수 편차(AI-교관) 평균 {aar['dev'].mean():+.1f}점, 절대 편차 {aar['dev'].abs().mean():.1f}점")

# 2) 왜 일어났나: 편차 원인 분해
aar["위험사건"] = aar[["near_collision", "hard_landing", "attitude_exceed"]].sum(axis=1).clip(0, 2).astype(str).str.replace("2", "2+")
print("\n[2. 왜 일어났나] 조건별 평균 편차:", aar.groupby("cond")["dev"].mean().round(1).to_dict())
ng = aar[aar["cond"] != "gust"]                                      # 조건 효과를 뺀 주간·야간 비행으로 비교
ev = ng.groupby("위험사건")["dev"].mean().round(1)
print("주간·야간 비행의 위험 사건 수별 평균 편차:", ev.to_dict())
print("주간·야간 비행의 습관별 평균 편차: 급조작", round(ng[ng["jerky"]]["dev"].mean(), 1), "/ 과보정", round(ng[ng["overcorr"]]["dev"].mean(), 1),
      "/ 습관 없음", round(ng[~ng["jerky"] & ~ng["overcorr"]]["dev"].mean(), 1))
print(f"-> 원인: (가) 주간 자료로만 맞춰 돌풍 비행을 평균 {aar[aar['cond'] == 'gust']['dev'].mean():.0f}점 과소평가(경로 이탈을 실력 부족으로 오인), "
      f"(나) 위험 사건 2건 이상 비행을 0건 대비 {ev.get('2+', np.nan) - ev.get('0', np.nan):+.1f}점 과대평가(장애물·착륙 대응 미반영), (다) 급조작 습관 미반영")

# 3) 무엇을 바꾸나: AAR 자료(3조건)로 격자 탐색, 기수 내 2겹 교차검증으로 선택
pid = aar["pilot"].str[1:].astype(int) % 2
grid = []
for name, feats in [("기본", BASE_F), ("고도화", ADV_F)]:
    for norm in (False, True):
        for alpha in (0.1, 1.0, 10.0):
            accs = []
            for k in (0, 1):
                tr, va = aar[pid != k], aar[pid == k]
                m = fit(tr, feats, norm, alpha)
                accs.append((to_grade(predict(m, va, tr, feats, norm)) == va["grade"]).mean())
            grid.append({"특징": name, "조건표준화": norm, "alpha": alpha, "교차검증 일치율": round(np.mean(accs), 3)})
g = pd.DataFrame(grid).sort_values("교차검증 일치율", ascending=False)
print("\n[3. 무엇을 바꾸나] 격자 12개 중 상위 4개")
print(g.head(4).to_string(index=False))
best = g.iloc[0]
TUNED = dict(feats=ADV_F if best["특징"] == "고도화" else BASE_F, norm=bool(best["조건표준화"]), alpha=float(best["alpha"]))
tuned_m = fit(aar, **TUNED)

# 4) 회귀 테스트: 새 기수 2개(학습에 쓰지 않은 조종자)
print("\n[회귀 테스트] 기수   초기 AI   재튜닝   변화")
reg, rep, near = [], [], []
for sd in (21, 22):
    new = make_dataset(40, seed=sd)
    a0 = to_grade(predict(init_m, new, day_train, INIT["feats"], INIT["norm"]))
    a1 = to_grade(predict(tuned_m, new, aar, TUNED["feats"], TUNED["norm"]))
    r0, r1 = (a0 == new["grade"]).mean(), (a1 == new["grade"]).mean()
    both = (a0 != new["grade"]) & (a1 != new["grade"])
    reg.append((r0, r1)); rep.append(int(both.sum()))
    near += list((np.minimum(np.abs(new["instructor"] - 55), np.abs(new["instructor"] - 75)) <= 5)[both])
    print(f"           #{sd}    {r0:6.1%}   {r1:6.1%}  {(r1 - r0) * 100:+5.1f}%p")
gain = np.mean([b - a for a, b in reg])
print(f"평균 향상 {gain * 100:+.1f}%p (목표 +10%p 이상: {'달성' if gain >= 0.10 else '미달'}), 회귀 테스트 {sum(b >= a for a, b in reg)}/{len(reg)} 통과")
print(f"재튜닝 후에도 반복된 불일치 {sum(rep)}건(전체 {2 * len(new)}회 중), 그중 교관 점수가 등급 경계(55·75점) ±5점 이내인 비행 {np.mean(near):.0%}")

w = pd.Series(tuned_m.coef_, index=TUNED["feats"]).round(2)
yaml = (f"features: [{', '.join(TUNED['feats'])}]\ncondition_normalization: {str(TUNED['norm']).lower()}\nridge_alpha: {TUNED['alpha']}\n"
        f"grade_cut: {{A: 75, B: 55}}\nweights: {{{', '.join(f'{k}: {v}' for k, v in w.items())}}}")
print("\n[재튜닝 파라미터(trn01_tuned_params.yaml 내용)]\n" + yaml)
print("지속 개선: 기수마다 교관 점수를 AAR 자료로 누적해 같은 절차로 재튜닝하고, 회귀 테스트 통과 시에만 교체한다.")

plt.figure(figsize=(7, 4))
for c, col in zip(COND, ["tab:blue", "tab:purple", "tab:orange"]):
    s = aar[aar["cond"] == c]
    plt.scatter(s["instructor"], s["dev"], s=12, c=col, label=c)
plt.axhline(0, c="k"); plt.xlabel("instructor score"); plt.ylabel("AI - instructor (initial AI)")
plt.legend(); plt.title("AAR: deviation of initial AI score (synthetic)"); plt.tight_layout(); plt.show()

# TODO: 재튜닝 파라미터를 configs/trn01_tuned_params.yaml 로, AAR 보고서를 docs/trn01_aar_report.md 로 저장하세요.


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[1. 무엇이 일어났나] 초기 AI 등급 일치율 47.5%, 점수 편차(AI-교관) 평균 -12.2점, 절대 편차 21.1점

[2. 왜 일어났나] 조건별 평균 편차: {'day': 2.7, 'gust': -40.1, 'night': 0.9}
주간·야간 비행의 위험 사건 수별 평균 편차: {'0': -9.8, '1': 0.6, '2+': 11.7}
주간·야간 비행의 습관별 평균 편차: 급조작 7.3 / 과보정 3.6 / 습관 없음 0.3
-> 원인: (가) 주간 자료로만 맞춰 돌풍 비행을 평균 -40점 과소평가(경로 이탈을 실력 부족으로 오인), (나) 위험 사건 2건 이상 비행을 0건 대비 +21.5점 과대평가(장애물·착륙 대응 미반영), (다) 급조작 습관 미반영

[3. 무엇을 바꾸나] 격자 12개 중 상위 4개
 특징  조건표준화  alpha  교차검증 일치율
고도화   True   10.0     0.825
고도화   True    0.1     0.817
고도화   True    1.0     0.808
고도화  False    0.1     0.783

[회귀 테스트] 기수   초기 AI   재튜닝   변화
           #21     45.0%    89.2%  +44.2%p
           #22     40.0%    81.7%  +41.7%p
평균 향상 +42.9%p (목표 +10%p 이상: 달성), 회귀 테스트 2/2 통과
재튜닝 후에도 반복된 불일치 21건(전체 240회 중), 그중 교관 점수가 등급 경계(55·75점) ±5점 이내인 비행 86%

[재튜닝 파라미터(trn01_tuned_params.yaml 내용)]
features: [path_rmse, hover_drift, hover_std, alt_err, obs_rt, obs_lateral, touch_vz, att_exceed, jerk, overshoot, n_risk]
condition_normalization: true
ridge_alpha: 10.0
grade_cut: {A: 75, B: 55}
weights: {path_rmse: -0.59, hover_drift: 1.22, hover_std: 0.77, alt_err: -7.67, obs_rt: -0.51, obs_lateral: 1.08, touch_vz: -4.19, att_exceed: -0.54, jerk: -1.59, overshoot: -0.66, n_risk: -2.4}
지속 개선: 기수마다 교관 점수를 AAR 자료로 누적해 같은 절차로 재튜닝하고, 회귀 테스트 통과 시에만 교체한다.
```
</details>
